# Portfolio Optimization Walkthrough

Step-by-step version of `main.py`: data → metrics → optimization → efficient frontier → out-of-sample backtest.

Set `DEMO = True` to run without internet (synthetic data, testing only).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
import plotly.express as px
from portfolio_engine import data, metrics, optimizer, backtest

DEMO = False
TICKERS = data.DEFAULT_TICKERS
BENCHMARK = 'SPY'
RF, MAX_W = 0.04, 0.30
pd.options.display.float_format = '{:.3f}'.format

## 1. Load prices and compute daily returns

In [ ]:
prices = data.generate_demo_prices(TICKERS) if DEMO else data.download_prices(TICKERS, '2018-01-01', cache_path=Path.cwd().parent / 'data' / 'prices.csv')
returns = data.to_returns(prices)
prices.tail()

## 2. Risk/return profile of each asset

Sharpe = excess return per unit of volatility. Beta = sensitivity to the benchmark. Max drawdown = worst peak-to-trough loss.

In [ ]:
metrics.summary_table(returns, returns[BENCHMARK], RF).sort_values('Sharpe', ascending=False)

In [ ]:
px.imshow(returns.corr(), text_auto='.2f', color_continuous_scale='RdBu_r', zmin=-1, zmax=1, title='Correlation')

## 3. Optimize: equal-weight vs. minimum-variance vs. maximum-Sharpe

Long-only, fully invested, each asset capped at `MAX_W`.

In [ ]:
weights = optimizer.build_portfolios(returns, RF, MAX_W)
weights.style.format('{:.1%}')

## 4. Efficient frontier

In [ ]:
mu, cov = optimizer.estimate_inputs(returns)
frontier = optimizer.efficient_frontier(mu.values, cov.values, max_weight=MAX_W)
cloud = optimizer.random_portfolios(mu.values, cov.values, rf=RF)
fig = px.scatter(cloud, x='Volatility', y='Return', color='Sharpe', opacity=0.4, title='Efficient frontier')
fig.add_scatter(x=frontier['Volatility'], y=frontier['Return'], mode='lines', name='Frontier', line=dict(color='black', width=3))
fig.update_layout(xaxis_tickformat='.0%', yaxis_tickformat='.0%')

## 5. Walk-forward backtest (the honest test)

The weights above were chosen with hindsight. Here each quarter's weights use only the prior 252 days, and 10 bps of costs are charged on turnover.

In [ ]:
oos, oos_w = backtest.walk_forward(returns, lookback=252, rebalance=63, rf=RF, max_weight=MAX_W, cost_bps=10)
oos['Benchmark'] = returns[BENCHMARK].loc[oos.index]
metrics.summary_table(oos, oos['Benchmark'], RF)

In [ ]:
px.line((1 + oos).cumprod(), title='Growth of $1 (out-of-sample, after costs)')

In [ ]:
px.area(oos_w['Max Sharpe'], title='Max Sharpe weights at each rebalance')

## 6. Takeaways

Write 2–3 sentences here after running with real data: which strategy held up out-of-sample, how concentrated Max Sharpe got, and how much the in-sample Sharpe overstated real-world performance.